In [3]:
import pickle
from utils.feedforward_neural_net_gpu import *

import io, torch

In [7]:
# force any tensor inside the pickle to land on CPU
torch.storage._load_from_bytes = lambda b: torch.load(
    io.BytesIO(b), map_location="cpu", weights_only=False
)

model_path = r"C:\Users\emper\Desktop\ICRA2026\locomotion_learning_information-theory\sim2real\model\ff_pos-act0_model_99.pickle"
with open(model_path, "rb") as file:
    loaded_data = pickle.load(file)

In [8]:
loaded_model = loaded_data[1]
solver = loaded_data[0]

In [9]:
# flat_model_param = loaded_model.get_a_model_params()
flat_model_param = solver.best_param()     # (71360,) 
architecture = loaded_model.architecture

print(architecture)
print(flat_model_param.shape)


[38, 128, 64, 19]
(14483,)


In [ ]:
temp_ff = FeedForwardNet(popsize=1, 
                       sizes=architecture, 
                       )
temp_ff.set_a_model_params(flat_model_param)


In [11]:
import json
import numpy as np

json_path = r"C:\Users\emper\Desktop\ICRA2026\locomotion_learning_information-theory\sim2real\model\ff_pos-act0_model_99.json"

payload = {
    "architecture": list(map(int, architecture)),
    "n_params": int(np.asarray(flat_model_param).size),
    "dtype": "float64",
    "source": "ff_pos-act0_model_99.pickle -> solver.best_param()",
    "flat_model_param": np.asarray(flat_model_param, dtype=np.float64).tolist(),
}

with open(json_path, "w") as f:
    json.dump(payload, f)

print("saved:", json_path)

saved: C:\Users\emper\Desktop\ICRA2026\locomotion_learning_information-theory\sim2real\model\ff_pos-act0_model_99.json


In [ ]:
# load back from JSON -- no pickle, no torch.load patch needed
with open(json_path) as f:
    cfg = json.load(f)

params = np.asarray(cfg["flat_model_param"], dtype=np.float64)

ff = FeedForwardNet(popsize=1,
                  sizes=cfg["architecture"],
                  )
ff.set_a_model_params(params)

# print("match:", np.array_equal(ff.get_a_model_params(), np.asarray(flat_model_param)))
print("match:", np.allclose(ff.get_a_model_params(), flat_model_param, atol=1e-6))

match: True
